# Step 2 — Spark submission through Livy

This notebook is a **read-only participant guide** with no executable cells. Read it in JupyterLab; perform the exercise separately in the organiser-approved terminal environment using `02_spark_via_livy.py`.

Complete Step 1 and confirm its cleanup first. Step 2 repeats the same bank-loan analysis through a different submission path, not a different dataset or a predictive model.

## Before you start

Wait for the organiser to confirm your assigned environment, approved Livy access and exact Step 2 submission command. Do not run with guessed connection settings or assume JupyterHub login authorises access to Livy.

**Validation scope:** separate organiser-owned HTTP client containers completed this analysis on both Tier-2 VMs, exported ten rows and passed owned-resource cleanup checks. Participant terminal access, concurrent-session capacity and failure-path recovery remain unvalidated.

The tested runner applies explicit session resource limits and verifies cleanup. The participant script's standalone entry point does not apply that same resource payload and only requests session deletion. The organiser must approve the participant launch path before use.


## 1. Reuse the dataset and inspect the local preview

Locate `02_spark_via_livy.py` and `data/Bank_Personal_Loan_Modelling.csv` in your assigned workspace. Reuse Step 1's field documentation and data-quality discussion; do not modify the input.

The script resolves local paths relative to itself. Its first activity prints column names and up to five local records without contacting Livy.

**Reflection:** which parts of this step can work even if the remote Spark services are unavailable?


## 2. Start the exercise through the approved Livy client

Open a JupyterLab terminal and use the **exact Step 2 command supplied by the organiser**. It must select the prepared client environment and run `02_spark_via_livy.py` or its approved bounded submission wrapper.

The script performs the analysis stages automatically. Do not execute notebook cells or launch another copy while a session may still exist.

### What changes from Step 1?

- **HTTP client:** the Python process in the approved terminal/client environment, not this guide.
- **Driver:** runs on the Livy side, rather than in the direct Spark submission client.
- **Master and worker executors:** provide the Spark cluster resources for the remote application.

The client sends statements over HTTP and receives their results. Variables such as the DataFrame exist in the remote session, not in a local notebook kernel.

### Prepared settings and access

`LIVY_URL` identifies the approved service. `LIVY_DATA_PATH` identifies the same CSV as seen by the remote driver and executors; it need not match the local preview path or Step 1's Spark path.

The current helper assumes organiser-approved access without additional API authentication. JupyterHub authentication does not itself secure Livy. Do not publish the endpoint, disable TLS verification, alter routing or guess addresses.


## 3. Understand session and statement readiness

Livy returns a **session ID** when a session is created. Creation does not mean the session is ready. The helper polls until the session becomes idle, then submits statements and waits for their output using the returned statement IDs.

Keep the printed session ID for reporting. Do not assume it is zero or reuse another participant's ID. Session IDs belong to their particular Livy service, so the same number on different VMs need not identify the same session.

**Expected output:** a created session ID, followed by the remote Spark version and master URL. The validated pilot uses Spark **3.5.3**; confirm the assigned master with the organiser.

Timeouts and remote errors are not successful completion. If creation times out without returning an ID, the service might still have created a session. Stop and ask the organiser to inspect it before retrying; do not guess an ID to delete.

**Reflection:** why must a client wait both for session readiness and for each statement's result?


## 4. Compare schema, filtering and category counts

Watch the terminal sections for the same operations as Step 1:

1. Read the CSV with a header and inferred schema; display five rows and count the input.
2. Select `Age`, `Income` and `Family`; filter with **Age > 30**; display up to ten rows and count the matches.
3. Group by `Personal Loan`, count each raw category and sort the summary.

**Expected results for the supplied snapshot, observed in both organiser-controlled Tier-2 tests:**

| Check | Result |
|---|---:|
| Complete input | 5,000 records |
| Age > 30 | 4,376 records |
| Personal Loan = 0 | 4,520 records |
| Personal Loan = 1 | 480 records |

These results should match Step 1 for the same snapshot. Preview order is not guaranteed. If counts differ, report the difference rather than changing the CSV.

Transformations build execution plans; actions request results. Interpret category codes using the dataset documentation, not assumptions.

**Reflection:** what stays the same in the analysis even though the driver and submission path have changed?


## 5. Locate the bounded result returned over HTTP

The remote driver collects at most ten filtered rows. The helper receives this bounded result as JSON through Livy and writes `outputs/02_filtered_preview.csv` locally beside the script's directory structure.

**Expected file:** header `Age,Income,Family`, followed by up to ten matching rows. Both organiser-controlled tests exported ten rows. An empty result would produce only the header.

A rerun replaces the Step 2 preview, but not Step 1's `filtered_preview.csv` or the input dataset. Download or rename a result before rerunning if it must be retained. Do not remove the row limit.

Follow the printed `Saved:` path and the organiser's output retrieval instructions. A path inside a client container is not automatically visible in your JupyterLab file browser.

**Reflection:** where is collection performed, and where is the CSV finally written?


## 6. Finish and distinguish a deletion request from verified cleanup

After a session ID has been obtained, the standalone script uses a `finally` block to attempt deletion of **that session only**, after success or an analysis failure.

**Normal script messages:** `Session deletion requested; cluster containers remain running.` followed by `Analysis completed successfully.`

An accepted deletion request does **not** prove that session teardown and application shutdown have finished. The tested organiser runner performs additional checks for session disappearance, application inactivity and owned-client removal. Do not assume the standalone script performs those checks.

If deletion fails, interruption occurs or completion is unclear, report the printed session ID, assigned Livy service and error to the organiser. If no ID was returned, report that explicitly. Closing a terminal or restarting JupyterHub does not guarantee deletion of a remote session.

Do not delete other sessions, stop shared containers or retry until the organiser confirms it is safe.


## Completion checklist

- [ ] I completed Step 1 and confirmed its cleanup.
- [ ] I used the organiser-approved Step 2 terminal submission path.
- [ ] I explained where the HTTP client, driver and executors run.
- [ ] I understood session IDs, statement IDs and readiness polling.
- [ ] I compared the schema and counts with Step 1.
- [ ] I located the bounded Step 2 preview.
- [ ] I distinguished a deletion request from confirmed cleanup.
- [ ] I reported any execution or cleanup uncertainty before retrying.

For missing settings, HTTP/TLS errors, timeouts, remote input-path failures or cleanup problems, stop and contact the organiser. Do not install dependencies, guess addresses or manage infrastructure.

Older guidance that asks you to run notebook cells does not apply to this Markdown-only guide.
